# B2-022 — Session 3: Autoencoders

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

An autoencoder learns to compress each input into a short code and rebuild the input from that code.
No labels are needed: the input is its own target.
This session builds the encoder/bottleneck/decoder shape ledger, trains with reconstruction (MSE) loss, separates undercomplete from overcomplete designs, and connects the linear autoencoder to Book 1 PCA through a projection-matrix distance between subspaces.
It ends with a fixed-seed training and held-out evaluation trace.

## 1. Encoder, bottleneck, decoder

An **autoencoder** is a pair of functions trained together:

* an **encoder** $f_\phi:\mathbb R^d\to\mathbb R^k$ that maps an input $x$ to a **code** (latent vector) $h=f_\phi(x)$;
* a **decoder** $g_\theta:\mathbb R^k\to\mathbb R^d$ that maps the code back to a **reconstruction** $\hat x=g_\theta(h)$.

The narrowest layer, of width $k$, is the **bottleneck**.
Training minimizes a reconstruction loss between $\hat x$ and $x$ itself, so the **reconstruction target is the input**: there is no separate label tensor, and a pipeline that feeds a class label as the target is not an autoencoder.

Why would copying the input be useful? Because the bottleneck forbids plain copying when $k<d$.
The network must discover a small set of coordinates that capture most of the variation in the data.
Those codes are useful as compressed features, for anomaly detection (inputs that reconstruct badly are unusual), for visualization when $k=2$, and as the starting point of the generative models in Sessions 4–5.

An autoencoder is **not** automatically a generative model.
Its decoder only knows how to decode codes that look like the codes the encoder produced; a random point in code space may decode to garbage.
Session 4 fixes that with a probabilistic code and a prior, which is the step from an autoencoder to a VAE.

**Checkpoint 1A.** An autoencoder receives a batch `x` of shape `(32,20)`. What tensor is its training target, and what shape does it have?

**Checkpoint 1B.** Why is an autoencoder with an untouched random-point decoder not yet a sampler of new data?

**Collected answers.** The target is `x` itself, shape `(32,20)`. Nothing forces codes to fill any known region, so decoding an arbitrary code has no guarantee of producing realistic data; a VAE adds a prior on codes to make sampling meaningful.

## 2. The MLP shape ledger

This unit uses multilayer-perceptron encoders and decoders only (Book 1 C5 `mlp-architecture`, `relu-activation`; C6 `nn-module`). Write the ledger before writing code. For an 8-feature input, hidden width 16, and code width 2:

| stage | operation | output shape | parameters |
|---|---|---|---:|
| input | — | `(B,8)` | — |
| encoder 1 | `Linear(8,16)` then ReLU | `(B,16)` | $8\cdot16+16=144$ |
| bottleneck | `Linear(16,2)` | `(B,2)` | $16\cdot2+2=34$ |
| decoder 1 | `Linear(2,16)` then ReLU | `(B,16)` | $2\cdot16+16=48$ |
| output | `Linear(16,8)` | `(B,8)` | $16\cdot8+8=136$ |

Total: $362$ parameters. Three conventions matter.
First, `nn.Linear(in, out)` stores `weight` with shape `(out, in)`, so `encoder[0].weight.shape == (16, 8)`.
Second, there is **no ReLU on the bottleneck output** and **no ReLU on the final output**: the code should be free to be negative, and the reconstruction must reach negative data values.
Third, expose `encode(x)` and `decode(h)` as separate methods, with `forward(x) = decode(encode(x))`, so later code can encode held-out rows or decode chosen codes.

```python
class MLPAutoencoder(nn.Module):
    def __init__(self, d=8, hidden=16, k=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, k))
        self.decoder = nn.Sequential(nn.Linear(k, hidden), nn.ReLU(), nn.Linear(hidden, d))
    def encode(self, x):
        return self.encoder(x)
    def decode(self, h):
        return self.decoder(h)
    def forward(self, x):
        return self.decode(self.encode(x))
```

**Checkpoint 2A.** Give the parameter count of a `6 -> 12 -> 3 -> 12 -> 6` MLP autoencoder with biases.

**Checkpoint 2B.** What goes wrong if a ReLU is placed after the final `Linear(16,8)` and the data has negative values?

**Collected answers.** $(6\cdot12+12)+(12\cdot3+3)+(3\cdot12+12)+(12\cdot6+6)=84+39+48+78=249$. Every negative target coordinate becomes unreachable (outputs are clipped at $0$), so the reconstruction error has a floor that training cannot remove.

## 3. Reconstruction loss and held-out reconstruction error

The standard reconstruction loss is Book 1 mean squared error, `F.mse_loss(x_hat, x)`, which averages $(\hat x_{ij}-x_{ij})^2$ over **all** $B\cdot d$ entries.
Two other reductions appear in the literature and in later sessions:

* **per-example sum over features**: $\sum_j(\hat x_{ij}-x_{ij})^2$ for each row $i$, shape `(B,)`; the VAE in Session 4 uses half of this;
* **batch mean of per-example sums**: equal to `F.mse_loss` multiplied by $d$.

For a plain autoencoder the choice only rescales the loss (which, with Adam, barely changes training), but it must be **stated**, because reported numbers are not comparable across reductions.
This unit's autoencoder practices pin the reduction: p17 uses `F.mse_loss` with its default mean reduction.

**Generalization is measured on held-out rows.** Use the immutable split from the unit generator: the model trains on the train rows and is evaluated, under `torch.no_grad()`, on held-out rows it never saw (Book 1 `train-test-split`).
A held-out reconstruction error that falls along with the training error certifies that the codes capture structure of the data source, not memorized rows.
A certificate states the **before** value (at initialization, step 0) and the **after** value, and compares them with a pre-stated threshold.
Evaluating on held-out rows does not leak information as long as their values never enter a `backward()` call, an optimizer step, a normalization statistic, or a model-selection decision.

**Checkpoint 3A.** For `x_hat - x` of shape `(4,8)` whose every entry is $0.5$, give `F.mse_loss` and the batch mean of per-example feature sums.

**Checkpoint 3B.** Name three ways held-out rows can leak into training even if they are never in the training batch.

**Collected answers.** `F.mse_loss` is $0.25$; the per-example sum is $8\cdot0.25=2$, and its batch mean is $2$. Through normalization statistics computed on all rows, through choosing hyperparameters or stopping time by held-out error, and through any `backward()`/optimizer step on a loss that includes them.

## 4. Undercomplete versus overcomplete

An autoencoder is **undercomplete** when $k<d$ and **overcomplete** when $k\ge d$.

**Undercomplete** models must compress. If the data truly lies near a $k$-dimensional surface, a good undercomplete model reconstructs well and its codes are coordinates on that surface.
If the data needs more than $k$ dimensions, reconstruction error stays high, which is itself information.

**Overcomplete** models can learn the identity map. Example: with $k=d$, a linear encoder and decoder whose weights are both the identity matrix (and whose biases are zero) give $\hat x=x$ exactly for every input, including inputs from a completely different distribution. Zero training error then certifies nothing about structure.
Overcomplete autoencoders only become useful with an extra constraint, such as noise added to the input (denoising), a sparsity penalty on codes, or the VAE's KL penalty.

Diagnostic habit: before training, ask what the trivial solution would be. For $k\ge d$ the trivial solution is the identity; for $k<d$ it is "predict the training mean," whose MSE equals the average per-feature variance. A trained undercomplete model must beat that mean baseline on held-out rows to show it learned anything.

**Checkpoint 4A.** Is a `10 -> 32 -> 12 -> 32 -> 10` autoencoder undercomplete or overcomplete?

**Checkpoint 4B.** What held-out MSE does the 'predict the training mean' baseline achieve, roughly, and why is it the right floor to beat?

**Collected answers.** Overcomplete: the code width $12$ is at least $d=10$. About the average per-feature variance of the data (using training means); any model that cannot beat it has learned nothing beyond the mean.

## 5. The linear autoencoder and PCA

A **linear autoencoder** has no activations: $h=W_e x+b_e$ and $\hat x=W_d h+b_d$ with `encoder = nn.Linear(d,k)` and `decoder = nn.Linear(k,d)`.
Every reconstruction $\hat x$ lies in the affine set $b_d+\operatorname{col}(W_d)$, a $k$-dimensional flat (when $W_d$ has full column rank $k$).

**Connection to Book 1 C9.** For centered data, PCA's top-$k$ subspace is the $k$-dimensional subspace that minimizes the mean squared distance from the data to its orthogonal projection (the C9 reconstruction view of PCA).
A linear autoencoder trained to the global minimum of MSE reaches that same best reconstruction, so the column space of its decoder weight is the **PCA subspace** of the training data (the biases absorb the mean, so centering is automatic).
We state this connection rather than prove it.

**What it does not give.** The autoencoder's decoder columns need not be orthonormal, need not be ordered by variance, and need not equal the PCA directions themselves: for any invertible $k\times k$ matrix $G$, replacing $W_d$ by $W_dG$ and $W_e$ by $G^{-1}W_e$ (and adjusting $b_e$) gives the same reconstructions.
So comparing weights entry by entry with PCA vectors is meaningless; only the **subspace** is identifiable.
The next section gives a tool that compares subspaces, and p18 uses it to certify that a trained linear autoencoder found the PCA subspace.

**Checkpoint 5A.** Why can't you certify a trained linear autoencoder by checking `decoder.weight` against the top PCA vectors with `torch.allclose`?

**Checkpoint 5B.** In a linear autoencoder, what plays the role of PCA's centering step?

**Collected answers.** The weights are identifiable only up to an invertible mixing $G$ (scale, sign, rotation within the subspace), so a correct model generally fails an entrywise comparison. The biases: $b_e$ and $b_d$ let the model reconstruct around the data mean.

## 6. Projection-matrix subspace distance

For a $d\times k$ matrix $W$ with linearly independent columns, define
$$P=W(W^\top W)^{-1}W^\top .$$
Facts (each checkable in one line): $P$ is symmetric; $P^2=P$; $PW=W$; and $Pv=0$ for every $v$ orthogonal to the columns of $W$. So $P$ is the **orthogonal projection** onto $\operatorname{col}(W)$.

**It depends only on the subspace.** For invertible $G$, using $(AB)^{-1}=B^{-1}A^{-1}$ (check: $B^{-1}A^{-1}AB=I$):
$$WG\big((WG)^\top WG\big)^{-1}(WG)^\top=WG\,G^{-1}(W^\top W)^{-1}G^{-\top}\,G^\top W^\top=P .$$

**Distance.** Two $k$-dimensional subspaces with projections $P_A,P_B$ are compared by the Frobenius norm $\lVert P_A-P_B\rVert_F$ (`torch.linalg.norm(PA - PB)`).
It is $0$ exactly when the subspaces coincide (if $P_A=P_B$ then $\operatorname{col}(W_A)=\operatorname{col}(P_A)=\operatorname{col}(P_B)=\operatorname{col}(W_B)$), and it is at most $\sqrt{2k}$.

**Computing it.** Never invert; solve. `W @ torch.linalg.solve(W.T @ W, W.T)` solves $(W^\top W)X=W^\top$ for the $k\times d$ matrix $X=(W^\top W)^{-1}W^\top$.

**Worked example.** $W_A=\begin{pmatrix}1&0\\0&1\\0&0\end{pmatrix}$ and $W_B=\begin{pmatrix}1&1\\1&-1\\0&0\end{pmatrix}$ span the same plane $\{x_3=0\}$, so the distance is $0$. $W_C=\begin{pmatrix}1&0\\0&0\\0&1\end{pmatrix}$ spans $\{x_2=0\}$: $P_A=\operatorname{diag}(1,1,0)$, $P_C=\operatorname{diag}(1,0,1)$, the difference is $\operatorname{diag}(0,1,-1)$, and the distance is $\sqrt2$.

**Checkpoint 6A.** What is $\lVert P_A-P_B\rVert_F$ for $W_A=(1,0)^\top$ and $W_B=(-3,0)^\top$ in $\mathbb R^2$?

**Checkpoint 6B.** Why does the formula need $W$ to have linearly independent columns?

**Collected answers.** $0$: both span the $x_1$-axis, and $P=\operatorname{diag}(1,0)$ for each. Otherwise $W^\top W$ is singular, the solve fails, and the projection must be built from a basis of the column space instead.

In [ ]:
import torch

def projector(W):
    return W @ torch.linalg.solve(W.T @ W, W.T)

WA = torch.tensor([[1., 0.], [0., 1.], [0., 0.]], dtype=torch.float64)
WB = torch.tensor([[1., 1.], [1., -1.], [0., 0.]], dtype=torch.float64)
WC = torch.tensor([[1., 0.], [0., 0.], [0., 1.]], dtype=torch.float64)
PA = projector(WA)
assert torch.allclose(PA @ PA, PA, atol=1e-12, rtol=1e-12)          # idempotent
assert torch.allclose(PA, PA.T, atol=1e-12, rtol=1e-12)             # symmetric
print("same plane:", torch.linalg.norm(PA - projector(WB)).item())
print("different planes:", torch.linalg.norm(PA - projector(WC)).item())

## 7. Worked laboratory: a fixed-seed tiny autoencoder trace

This lesson-only laboratory uses twelve literal training rows and four literal held-out rows on a curved one-dimensional path in $\mathbb R^4$: $x(t)=(t,\;t^2-1,\;0.5t,\;1-t)$.
The rows span only a 2-dimensional linear space, but they lie on a 1-dimensional **curve**, so a nonlinear code of width $k=1$ can describe them while a linear (PCA) code of width $1$ cannot.
It is not the unit generator's data and not p17's model or answer.

**Protocol.** Seed `20260927` immediately before building `Sequential(Linear(4,8), ReLU, Linear(8,1), Linear(1,8), ReLU, Linear(8,4))`; Adam with `lr=0.02`; 300 full-batch updates in `zero_grad(set_to_none=True)` → forward → `F.mse_loss` → `backward` → `step` order; evaluate train and held-out MSE under `no_grad` at the listed steps.

| update | train MSE | held-out MSE |
|---:|---:|---:|
| 0 | 0.9802 | 0.6541 |
| 1 | 0.9195 | 0.6101 |
| 50 | 0.0135 | 0.0132 |
| 100 | 0.0022 | 0.0038 |
| 300 | 0.0009 | 0.0014 |

For reference, the best **linear** width-1 reconstruction (PCA with one component, fitted on the training rows) has held-out MSE $0.1302$. The nonlinear bottleneck beats it by about $90\times$ because it can bend along the curve.

How to read the trace. Update 0 is the baseline before any step. One update changes little, so a single step is not a learning certificate. Held-out MSE falls with training MSE, so the code generalizes. Only the twelve training rows ever entered `backward()`; the four held-out rows were evaluated only. A certificate would state in advance, for example, "held-out MSE after 300 updates is at most one tenth of held-out MSE at update 0," and then check it.

The second cell trains a **linear** width-1 autoencoder on eight literal 3-D rows and certifies that its decoder column spans the same line as PCA's first direction, using the Section 6 distance: it starts near $1.33$ and falls to about $0.0006$ after 800 Adam updates, even though the learned weight is a rescaled, sign-flipped version of the PCA vector.

**Checkpoint 7A.** In the trace, how many times smaller is held-out MSE at update 300 than at update 0?

**Checkpoint 7B.** The linear-AE cell's decoder weight is about $(1.02,0.52,-0.004)$ and PCA's direction about $(-0.89,-0.45,0.003)$. Why is the subspace distance nevertheless near $0$?

**Collected answers.** $0.6541/0.0014\approx467$ times smaller. The two vectors are (approximately) scalar multiples of each other, $G\approx-1.15$, and the projection depends only on the spanned line, not the scale or sign.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(1)

def curve(ts):
    t = torch.tensor(ts, dtype=torch.float32).unsqueeze(1)
    return torch.cat([t, t ** 2 - 1.0, 0.5 * t, 1.0 - t], dim=1)

train_x = curve([-1.5, -1.2, -0.9, -0.6, -0.3, 0.0, 0.3, 0.6, 0.9, 1.2, 1.5, 1.8])
heldout_x = curve([-1.05, -0.15, 0.45, 1.35])
torch.manual_seed(20260927)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1),
                      nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 4))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

def evaluate():
    with torch.no_grad():
        return F.mse_loss(model(train_x), train_x).item(), F.mse_loss(model(heldout_x), heldout_x).item()

trace = {0: evaluate()}
for step in range(1, 301):
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(train_x), train_x)
    loss.backward()
    optimizer.step()
    if step in (1, 50, 100, 300):
        trace[step] = evaluate()
for step, (train_mse, heldout_mse) in trace.items():
    print(f"update {step:>3}: train {train_mse:.4f}  held-out {heldout_mse:.4f}")
assert trace[300][1] <= 0.1 * trace[0][1]

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

def projector(W):
    return W @ torch.linalg.solve(W.T @ W, W.T)

rows = torch.tensor([[2.0, 1.0, 0.1], [-1.0, -0.4, 0.0], [0.5, 0.4, -0.1], [-2.0, -1.1, 0.1],
                     [1.0, 0.3, 0.0], [-0.5, 0.0, -0.1], [3.0, 1.6, 0.0], [0.0, -0.2, 0.1]])
_, _, Vh = torch.linalg.svd(rows - rows.mean(dim=0), full_matrices=False)
p_pca = projector(Vh[:1].T)                                   # PCA line of the centered rows
torch.manual_seed(20260927)
linear_ae = nn.Sequential(nn.Linear(3, 1), nn.Linear(1, 3))   # encoder, decoder
optimizer = torch.optim.Adam(linear_ae.parameters(), lr=0.02)
print("start distance:", round(torch.linalg.norm(projector(linear_ae[1].weight.detach()) - p_pca).item(), 4))
for step in range(800):
    optimizer.zero_grad(set_to_none=True)
    F.mse_loss(linear_ae(rows), rows).backward()
    optimizer.step()
distance = torch.linalg.norm(projector(linear_ae[1].weight.detach()) - p_pca).item()
print("final distance:", round(distance, 5), " decoder column:", linear_ae[1].weight.detach().flatten().tolist())
assert distance < 0.01

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Wrong target.* Training against labels or against the code instead of the input. Fix: `loss = F.mse_loss(model(x), x)`.
2. *Activation on the output or the code.* A final ReLU clips negative data. Fix: linear output layer; linear bottleneck.
3. *Overcomplete identity.* Celebrating near-zero training error with $k\ge d$. Fix: compare with held-out error and with a constrained (undercomplete or regularized) model.
4. *Comparing weights instead of subspaces.* Fix: projection matrices and $\lVert P_A-P_B\rVert_F$.
5. *Uncentered PCA.* Computing PCA on raw rows when the linear autoencoder has biases. Fix: subtract the training mean before the SVD.
6. *Leaky evaluation.* Normalizing with all-row statistics or picking the step count by held-out error. Fix: fit everything on train rows only; evaluate held-out once at the pre-stated protocol's end (and at step 0 as the baseline).
7. *Unstated reduction.* Reporting an MSE without saying mean-over-entries versus per-example sum. Fix: name the reduction in the certificate.

**Exam connections.** p04 asks for the bottleneck shape and the reconstruction target; p10 asks you to build an MLP autoencoder whose shape ledger and parameter count are pinned; p17 trains a tiny autoencoder and certifies held-out improvement; p18 trains a linear autoencoder and certifies the PCA subspace with the projection distance; p21 asks when an autoencoder is the right tool and when a VAE is needed.

**Checkpoint 8A.** A linear autoencoder certificate reports distance $0.03$ from the PCA subspace computed on **uncentered** rows. What is suspect?

**Checkpoint 8B.** Which practice pins the shape ledger and parameter count of an MLP autoencoder?

**Collected answers.** The reference subspace is wrong: with biases, the autoencoder targets the subspace of centered data, so PCA must use centered training rows (and the distance should be recomputed). p10.

## 9. Forward-only deeper topic: denoising and sparse autoencoders

Forward-only deeper topic.
A **denoising autoencoder** corrupts each input with noise and trains the decoder to output the clean input; this prevents the overcomplete identity solution and teaches the model which directions are "data" and which are "noise."
Learning to remove Gaussian noise step by step is the central idea of the diffusion models in `B2-023-generative-models-diffusion`.
A **sparse autoencoder** penalizes code activations so that each input uses few code units; sparse autoencoders are also used to interpret large models.
Convolutional autoencoders for images are out of scope for this unit, which uses MLPs only.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-022.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Is a denoising objective required in p17?

**Checkpoint 9B.** Which later unit builds on learning to remove Gaussian noise?

**Collected answers.** No; p17 uses the plain reconstruction protocol in its statement. `B2-023-generative-models-diffusion`.